# Silver_311_v1 — NYC 311 Service Requests: Bronze → Silver Cleaning
### Task 4.3 | Notebook 3 | `stg_nyc_311`

This notebook ingests the raw NYC 311 Service Request data from the **Bronze** layer and produces
a clean, validated, and SCD-tracked **Silver Delta table** (`silver_stg_nyc_311`).

**16 cleaning steps** are applied covering null rejection, deduplication, type casting, column pruning,
date validation, status harmonisation, routing-code removal, text standardisation, business-rule validation,
coordinate checking, and multi-strategy recovery of Borough, Park Facility Name, and Council District.

Rows that cannot be automatically repaired land in a **Silver quarantine table** (`silver_quarantine_311`)
for manual review. Source-level duplicates go to a separate **Bronze quarantine table**.

Three **pre-flight checks** run before any cleaning step:
- **Pre-Step A** — Schema drift detection
- **Pre-Step B** — Row-count reconciliation against a source manifest
- **Pre-Step C** — Data-quality metrics framework (accumulates counts per step; written to Silver at the end)


---
## Imports, Spark Session & Configuration

In [25]:
%pip install geopandas shapely pyproj fiona --quiet

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 36, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [26]:
# ── Standard library ──────────────────────────────────────────────────────────
from collections import defaultdict
from datetime import date
import json

# ── PySpark ───────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ── Geospatial (used in Steps 11-13) ─────────────────────────────────────────
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

spark = SparkSession.builder.getOrCreate()

# ── PATH CONFIGURATION ────────────────────────────────────────────────────────
# All paths use Fabric relative-path format (Files/...) consistent with the
# Bronze ingestion notebooks. No ABFS paths needed when running inside Fabric.
LAKEHOUSE  = "test_lakehouse_1"
BASE_PATH  = "Files/project_data"

# Source: read directly from the bronze Delta table (registered in metastore)
# Pre-Step A will compare its columns against EXPECTED_COLUMNS.
BRONZE_TABLE = "bronze_311_nyc"

# Silver output paths — Delta tables written here and registered in the metastore
SILVER_TABLE_PATH      = f"{BASE_PATH}/silver/silver_stg_nyc_311"
SILVER_QUARANTINE_PATH = f"{BASE_PATH}/silver/silver_quarantine_311"
BRONZE_QUARANTINE_PATH = f"{BASE_PATH}/bronze/bronze_quarantine_311_silver_dupes"
DQ_METRICS_PATH        = f"{BASE_PATH}/silver/dq_metrics_311"

# Reference GeoJSON files — upload these to your Lakehouse Files before running
# Steps 11-13. The paths below match the expected upload location.
BOROUGH_GEOJSON_PATH = f"{BASE_PATH}/reference/borough_boundaries.geojson"
PARKS_GEOJSON_PATH   = f"{BASE_PATH}/reference/parks_properties.geojson"
COUNCIL_GEOJSON_PATH = f"{BASE_PATH}/reference/council_districts.geojson"

# ── TABLE NAMES (registered in the Fabric metastore) ─────────────────────────
SILVER_TABLE     = "silver_stg_nyc_311"
QUARANTINE_TABLE = "silver_quarantine_311"

# ── RUN METADATA ──────────────────────────────────────────────────────────────
TODAY = date.today().isoformat()

# ── DQ METRICS ACCUMULATOR ────────────────────────────────────────────────────
# This dict is populated by every step and written to Delta at the end.
# It MUST be initialized here before Pre-Step A runs.
dq_metrics = {}

# ── KNOWN VALID DOMAIN VALUES ─────────────────────────────────────────────────
VALID_AGENCIES = frozenset({
    "NYPD", "HPD", "DSNY", "DOT", "DEP", "DPR", "DOB",
    "TLC", "DHS", "DOHMH", "FDNY", "HRA", "DOF", "DCWP", "DCA"
})

VALID_STATUSES = frozenset({
    "Open", "Closed", "In Progress", "Assigned",
    "Pending", "Unspecified", "Started"
})

BOROUGH_BBL_MAP = {
    "1": "Manhattan",
    "2": "Bronx",
    "3": "Brooklyn",
    "4": "Queens",
    "5": "Staten Island"
}

# Expected columns from the bronze_311_nyc Delta table (post-Bronze ingestion).
# Column names are lowercase with underscores — as written by the Bronze notebook.
# Pre-Step A compares the actual table columns against this set.
EXPECTED_COLUMNS = {
    "unique_key", "created_date", "closed_date", "agency", "agency_name",
    "complaint_type", "descriptor", "location_type", "incident_zip",
    "incident_address", "street_name", "cross_street_1", "cross_street_2",
    "intersection_street_1", "intersection_street_2", "address_type",
    "city", "landmark", "facility_type", "status", "community_board",
    "bbl", "borough", "x_coordinate_state_plane", "y_coordinate_state_plane",
    "park_facility_name", "park_borough", "latitude", "longitude", "location",
    "due_date", "resolution_description", "resolution_action_updated_date",
    "taxi_company_borough", "taxi_pick_up_location", "bridge_highway_name",
    "bridge_highway_direction", "road_ramp", "bridge_highway_segment",
    "vehicle_type", "police_precinct", "council_district",
    "source", "ingest_run_id", "ingested_at", "ingest_date",
}

print(f"Configuration loaded. Run date: {TODAY}")
print(f"Bronze source table : {BRONZE_TABLE}")
print(f"Silver output path  : {SILVER_TABLE_PATH}")


StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 38, Finished, Available, Finished, False)

Configuration loaded. Run date: 2026-04-20
Bronze source table : bronze_311_nyc
Silver output path  : Files/project_data/silver/silver_stg_nyc_311


---
## Pre-Step A — Schema Drift Detection

The very first thing we do — before a single row is touched — is compare the
columns that arrived from the Bronze source against the 44 columns we expect
from the NYC 311 Open Data export.

| Scenario | Action |
|---|---|
| **Missing column(s)** | Pipeline aborts with a descriptive error. Downstream casts and validations will silently break or produce wrong results if a column is absent. |
| **Extra column(s)** | Warning only. NYC may have added new fields we haven't mapped yet. The unknown columns flow through unchanged but are not validated. |
| **Exact match** | PASS — proceed normally. |

This guard prevents the pipeline from producing corrupt Silver data when NYC
changes its export schema without notice.

In [27]:
# ── Load Bronze Delta table ───────────────────────────────────────────────────
df_raw = spark.table(BRONZE_TABLE)

# ── Rename columns: Bronze uses snake_case; this notebook uses Title Case ─────
RENAME_MAP = {
    "unique_key":                       "Unique Key",
    "created_date":                     "Created Date",
    "closed_date":                      "Closed Date",
    "agency":                           "Agency",
    "agency_name":                      "Agency Name",
    "complaint_type":                   "Complaint Type",
    "descriptor":                       "Descriptor",
    "location_type":                    "Location Type",
    "incident_zip":                     "Incident Zip",
    "incident_address":                 "Incident Address",
    "street_name":                      "Street Name",
    "cross_street_1":                   "Cross Street 1",
    "cross_street_2":                   "Cross Street 2",
    "intersection_street_1":            "Intersection Street 1",
    "intersection_street_2":            "Intersection Street 2",
    "address_type":                     "Address Type",
    "city":                             "City",
    "landmark":                         "Landmark",
    "facility_type":                    "Facility Type",
    "status":                           "Status",
    "community_board":                  "Community Board",
    "bbl":                              "BBL",
    "borough":                          "Borough",
    "x_coordinate_state_plane":         "X Coordinate (State Plane)",
    "y_coordinate_state_plane":         "Y Coordinate (State Plane)",
    "park_facility_name":               "Park Facility Name",
    "park_borough":                     "Park Borough",
    "latitude":                         "Latitude",
    "longitude":                        "Longitude",
    "location":                         "Location",
    "due_date":                         "Due Date",
    "resolution_description":           "Resolution Description",
    "resolution_action_updated_date":   "Resolution Action Updated Date",
    "taxi_company_borough":             "Taxi Company Borough",
    "taxi_pick_up_location":            "Taxi Pick Up Location",
    "bridge_highway_name":              "Bridge Highway Name",
    "bridge_highway_direction":         "Bridge Highway Direction",
    "road_ramp":                        "Road Ramp",
    "bridge_highway_segment":           "Bridge Highway Segment",
    "vehicle_type":                     "Vehicle Type",
    "police_precinct":                  "Police Precinct",
    "council_district":                 "Council District",
}

for old, new in RENAME_MAP.items():
    if old in df_raw.columns:
        df_raw = df_raw.withColumnRenamed(old, new)

# ── Expected columns in Title Case (after rename) ─────────────────────────────
EXPECTED_COLUMNS = {
    "Unique Key", "Created Date", "Closed Date", "Agency", "Agency Name",
    "Complaint Type", "Descriptor", "Location Type", "Incident Zip",
    "Incident Address", "Street Name", "Cross Street 1", "Cross Street 2",
    "Intersection Street 1", "Intersection Street 2", "Address Type",
    "City", "Landmark", "Facility Type", "Status", "Community Board",
    "BBL", "Borough", "X Coordinate (State Plane)", "Y Coordinate (State Plane)",
    "Park Facility Name", "Park Borough", "Latitude", "Longitude", "Location",
    "Due Date", "Resolution Description", "Resolution Action Updated Date",
    "Taxi Company Borough", "Taxi Pick Up Location", "Bridge Highway Name",
    "Bridge Highway Direction", "Road Ramp", "Bridge Highway Segment",
    "Vehicle Type", "Police Precinct", "Council District",
}

# ── Schema drift detection ─────────────────────────────────────────────────────
actual_columns  = set(df_raw.columns)
missing_columns = EXPECTED_COLUMNS - actual_columns
extra_columns   = actual_columns   - EXPECTED_COLUMNS

drift_status = "PASS" if not missing_columns and not extra_columns else "WARN"

dq_metrics["pre_A_schema_drift"] = {
    "expected_column_count": len(EXPECTED_COLUMNS),
    "actual_column_count":   len(actual_columns),
    "missing_columns":       sorted(missing_columns),
    "extra_columns":         sorted(extra_columns),
    "status":                drift_status,
}

if missing_columns:
    print(
        f"[SCHEMA DRIFT WARNING] {len(missing_columns)} expected column(s) not found: "
        f"{sorted(missing_columns)}\n"
        f"Steps that reference these columns may fail or produce nulls. "
        f"Check the Bronze table schema."
    )

if extra_columns:
    print(
        f"[SCHEMA DRIFT WARNING] {len(extra_columns)} unrecognised column(s): "
        f"{sorted(extra_columns)}\n"
        f"These pass through unchanged but are not validated."
    )

print(f"Pre-Step A (Schema Drift): {drift_status} — "
      f"{len(actual_columns)} columns found, "
      f"{len(missing_columns)} missing, {len(extra_columns)} extra.")

df = df_raw

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 39, Finished, Available, Finished, False)

[SCHEMA DRIFT WARNING] 1 expected column(s) not found: ['Location']
Steps that reference these columns may fail or produce nulls. Check the Bronze table schema.
[SCHEMA DRIFT WARNING] 7 unrecognised column(s): ['descriptor_2', 'ingest_date', 'ingest_run_id', 'ingested_at', 'location_coordinates', 'open_data_channel_type', 'source']
These pass through unchanged but are not validated.
Pre-Step A (Schema Drift): WARN — 48 columns found, 1 missing, 7 extra.


---
## Pre-Step B — Row-Count Reconciliation

Record how many rows arrived from Bronze **before any cleaning**.  
This baseline becomes the accountability ledger for the whole pipeline:
every row that leaves `df` must appear in exactly one of:
- `silver_stg_nyc_311` (clean), or
- `silver_quarantine_311` (needs review), or
- the hard-rejected set (permanently dropped with a reason).

If your Bronze ingestion step writes a manifest file with the expected row
count, set `SOURCE_ROW_COUNT_FROM_MANIFEST` to that value.  
If not, the check simply records the ingested count for cross-run comparison.

In [28]:
SOURCE_ROW_COUNT_FROM_MANIFEST = None   # set to int if manifest is available

ingested_row_count = df.count()

manifest_match = (
    SOURCE_ROW_COUNT_FROM_MANIFEST is None
    or ingested_row_count == SOURCE_ROW_COUNT_FROM_MANIFEST
)

dq_metrics["pre_B_row_count"] = {
    "ingested_row_count": ingested_row_count,
    "manifest_count":     SOURCE_ROW_COUNT_FROM_MANIFEST,
    "match":              manifest_match,
}

if SOURCE_ROW_COUNT_FROM_MANIFEST and not manifest_match:
    delta = ingested_row_count - SOURCE_ROW_COUNT_FROM_MANIFEST
    print(
        f"[ROW COUNT WARNING] Ingested {ingested_row_count:,} rows but manifest "
        f"says {SOURCE_ROW_COUNT_FROM_MANIFEST:,}. Delta: {delta:+,}."
    )
else:
    print(f"Pre-Step B (Row Count): {ingested_row_count:,} rows ingested from Bronze.")

# ── Accumulators ──────────────────────────────────────────────────────────────
hard_rejected_frames = []   # permanently dropped rows (no recovery path)
quarantine_frames    = []   # rows for manual review


StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 40, Finished, Available, Finished, False)

Pre-Step B (Row Count): 1,207,334 rows ingested from Bronze.


---
## Pre-Step C — Data-Quality Metrics Framework

Two helper functions are defined here and called after every cleaning step:

- **`record_step_metrics`** — snapshots the clean / quarantined / rejected row
  counts into `dq_metrics` and prints a summary line.
- **`tag_quarantine`** — stamps a quarantine reason and today's date onto a
  DataFrame before it is written to the quarantine table.

At the very end of the notebook, `dq_metrics` is serialised as a JSON record
and written to the Silver layer so pipeline health can be monitored over time.

In [29]:
def record_step_metrics(
    step_name: str,
    df_clean,
    df_quarantined=None,
    df_rejected=None,
    notes: str = ""
):
    """
    Snapshot row counts into dq_metrics after a cleaning step.

    Parameters
    ----------
    step_name      : Key used in the dq_metrics dict (e.g. "step_01_critical_nulls")
    df_clean       : The main DataFrame after the step
    df_quarantined : Rows sent to the quarantine table this step (optional)
    df_rejected    : Rows permanently rejected this step (optional)
    notes          : Free-text annotation added to the metrics record
    """
    n_clean      = df_clean.count()
    n_quarantine = df_quarantined.count() if df_quarantined is not None else 0
    n_rejected   = df_rejected.count()   if df_rejected   is not None else 0

    dq_metrics[step_name] = {
        "rows_clean":       n_clean,
        "rows_quarantined": n_quarantine,
        "rows_rejected":    n_rejected,
        "notes":            notes,
    }

    print(
        f"[{step_name}]  clean={n_clean:,}  "
        f"quarantined={n_quarantine:,}  rejected={n_rejected:,}"
        + (f"  ▸ {notes}" if notes else "")
    )


def tag_quarantine(df, reason: str):
    """
    Stamp a quarantine reason and date onto a DataFrame.
    These two columns are expected by the quarantine table schema.
    """
    return (
        df.withColumn("quarantine_reason", F.lit(reason))
          .withColumn("quarantine_date",   F.lit(TODAY))
    )

print("Pre-Step C: DQ metrics helpers ready.")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 41, Finished, Available, Finished, False)

Pre-Step C: DQ metrics helpers ready.


---
## Step 1 — Critical Field Null Check

Three fields are critical for the pipeline's downstream logic:

| Field | Why it is critical | Action if NULL |
|---|---|---|
| **Unique Key** | The SCD MERGE key in Step 15. Without it there is no way to match incoming rows against the Silver table. | Hard reject — row is permanently dropped |
| **Created Date** | Forms the composite join key `(borough, hour)` used in the Gold layer. Also anchors all date-based validations in Step 5. | Hard reject — row is permanently dropped |
| **Borough** | The other half of the composite Gold join key. Has a recovery path via Step 11 (Park Borough, BBL, city mapping, geo-lookup). | Flag for recovery — row is tagged and flows through all cleaning steps; if Step 11 cannot recover it, it goes to quarantine |

> **Note:** NULL-Borough rows are **not split out** into a separate DataFrame.
> They stay in `df` alongside all other rows and go through every cleaning step.
> A boolean flag column `_pending_borough_recovery` marks them so Step 11 can
> target them precisely without duplicating any processing logic.


In [30]:
# ── Hard reject: NULL / empty Unique Key or Created Date ─────────────────────
# Empty strings are treated the same as NULL because the source CSV sometimes
# exports blank cells as "" rather than a true NULL value.

hard_reject_mask = (
    F.col("Unique Key").isNull()
    | (F.trim(F.col("Unique Key")) == "")
    | F.col("Created Date").isNull()
    | (F.trim(F.col("Created Date")) == "")
)

df_rejected_step1 = (
    df.filter(hard_reject_mask)
      .withColumn("rejection_reason", F.lit("NULL_UNIQUE_KEY_OR_CREATED_DATE"))
)

df = df.filter(~hard_reject_mask)
hard_rejected_frames.append(df_rejected_step1)

# ── Flag: NULL / empty Borough (recovery attempted in Step 11) ────────────────
# Rows flagged True here will be targeted by the Borough recovery logic in
# Step 11. After Step 11, any rows still flagged go to the quarantine table.

null_borough_mask = (
    F.col("Borough").isNull()
    | (F.trim(F.col("Borough")) == "")
    | (F.upper(F.trim(F.col("Borough"))) == "UNSPECIFIED")
)

df = df.withColumn("_pending_borough_recovery", null_borough_mask)

n_null_borough = df.filter(F.col("_pending_borough_recovery")).count()

dq_metrics["step_01_critical_nulls"] = {
    "hard_rejected":              df_rejected_step1.count(),
    "flagged_for_borough_recovery": n_null_borough,
    "rows_remaining_in_df":       df.count(),
}

print(
    f"[Step 1] Hard-rejected: {df_rejected_step1.count():,}  |  "
    f"Flagged for Borough recovery: {n_null_borough:,}  |  "
    f"Clean rows: {df.count():,}"
)

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 42, Finished, Available, Finished, False)

[Step 1] Hard-rejected: 0  |  Flagged for Borough recovery: 1,068  |  Clean rows: 1,207,334


---
## Step 2 — Remove Duplicate Records

We identify rows in the incoming Bronze file that share the same `Unique Key`.
These are **source-side duplicates** — the same complaint exported more than
once by NYC's system — and are distinct from the intentional multi-row history
handled in Step 15 (status changes over time).

**Tie-breaking strategy** (applied in order):
1. `Created Date` descending — the most-recently-dated copy wins.
2. `Closed Date`  descending, nulls last — if Created Date ties, prefer the
   row that carries a Closed Date (it holds more information).

Losing copies go to the **Bronze quarantine table** for independent review.
They are kept separate from the Silver quarantine so it is always clear
whether a flagged row originated from a source defect or a cleaning decision.

In [31]:
window_dedup = (
    Window
    .partitionBy("Unique Key")
    .orderBy(
        F.col("Created Date").desc(),
        F.col("Closed Date").desc_nulls_last()
    )
)

df = df.withColumn("_dedup_rank", F.row_number().over(window_dedup))

df_bronze_dupes = (
    df.filter(F.col("_dedup_rank") > 1)
      .drop("_dedup_rank")
      .withColumn("quarantine_reason", F.lit("DUPLICATE_UNIQUE_KEY"))
      .withColumn("quarantine_date",   F.lit(TODAY))
)

df = df.filter(F.col("_dedup_rank") == 1).drop("_dedup_rank")

n_dupes = df_bronze_dupes.count()

dq_metrics["step_02_deduplication"] = {
    "duplicates_found": n_dupes,
    "rows_remaining":   df.count(),
    "notes": "Duplicates written to bronze_quarantine_311_silver_dupes. Tiebreaker: Closed Date desc nulls last."
}

if n_dupes > 0:
    (
        df_bronze_dupes.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .partitionBy("quarantine_date")
        .saveAsTable("bronze_quarantine_311_silver_dupes")
    )
    print(f"[Step 2] {n_dupes:,} duplicate(s) → bronze_quarantine_311_silver_dupes.")
else:
    print("[Step 2] No duplicates found.")

print(f"[Step 2] Rows remaining: {df.count():,}")


StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 43, Finished, Available, Finished, False)

[Step 2] No duplicates found.
[Step 2] Rows remaining: 1,207,334


---
## Step 3 — Cast Data Types

PySpark reads every column of a CSV as `StringType` when `inferSchema=false`.
We now apply explicit casts to every column that is not semantically a string.
Doing this **after** deduplication (Step 2) but **before** any value-based
logic ensures that all downstream comparisons operate on the correct types.

| Column(s) | Raw format | Target type | Notes |
|---|---|---|---|
| `Created Date`, `Closed Date`, `Resolution Action Updated Date` | `MM/DD/YYYY HH:MM:SS AM/PM` | `Timestamp` | `to_timestamp` returns NULL for blank Closed Date — correct for open complaints |
| `Incident Zip` | `float` e.g. `1001.0` | `String` e.g. `"01001"` | Strip `.0`, left-pad to 5 chars |
| `Council District` | `float` e.g. `12.0` | `Integer` | Strip decimal |
| `BBL` | `float` e.g. `3012340056.0` | `String` e.g. `"3012340056"` | Property identifier, not a number |
| `Latitude`, `Longitude` | `string` → numeric comparison needed | `Double` | NULL preserved for missing coords |

In [32]:
# ── Step 3: Type enforcement ──────────────────────────────────────────────────
# The Bronze table already has cast types from the ingestion notebook.
# This step re-enforces types to guard against schema drift or future
# Bronze changes. Casting a column to its existing type is a safe no-op.

df = (
    df
    # Timestamps — already cast by Bronze; this is a safety re-enforcement
    .withColumn("Created Date",                  F.col("Created Date").cast(T.TimestampType()))
    .withColumn("Closed Date",                   F.col("Closed Date").cast(T.TimestampType()))
    .withColumn("Resolution Action Updated Date",F.col("Resolution Action Updated Date").cast(T.TimestampType()))

    # Incident Zip: ensure 5-char zero-padded string (Bronze stores as string already)
    .withColumn("Incident Zip",
        F.regexp_replace(
            F.lpad(
                F.regexp_replace(F.col("Incident Zip").cast("string"), r"\.0$", ""),
                5, "0"
            ),
            r"^0+$", "00000"
        )
    )

    # Council District: integer
    .withColumn("Council District", F.col("Council District").cast(T.IntegerType()))

    # BBL: strip any trailing .0 from string representation
    .withColumn("BBL", F.regexp_replace(F.col("BBL").cast("string"), r"\.0$", ""))

    # Coordinates: double
    .withColumn("Latitude",  F.col("Latitude").cast(T.DoubleType()))
    .withColumn("Longitude", F.col("Longitude").cast(T.DoubleType()))
)

dq_metrics["step_03_type_casting"] = {
    "rows_after_cast": df.count(),
    "notes": "Types re-enforced from Bronze Delta table. No rows dropped."
}

print(f"[Step 3] Type enforcement complete. Rows: {df.count():,}")
print(f"         Closed Date NULL (open complaints): {df.filter(F.col('Closed Date').isNull()).count():,}")


StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 44, Finished, Available, Finished, False)

[Step 3] Type enforcement complete. Rows: 1,207,334
         Closed Date NULL (open complaints): 104,025


---
## Step 4 — Drop Low-Value and Redundant Columns

We drop columns that are either almost entirely empty, carry redundant
information, or are structurally too inconsistent to be reliably used for
analysis. **Nothing is dropped from Bronze** — the raw 44-column file is
permanent. We only prune the Silver working DataFrame.

Two groups of columns (`Bridge/Highway` and `TLC-specific`) are conditionally
dropped: if their completeness within the relevant complaint type is ≥ 20 %,
they are kept in Silver for specialised analysis. Otherwise they are dropped.

> **`Park Borough` is deliberately kept at this stage.**
> It is used as the first fallback in the Borough recovery logic in Step 11.
> It will be dropped from Silver after Step 11 completes.

In [33]:
# ── Unconditional drops ───────────────────────────────────────────────────────

UNCONDITIONAL_DROPS = [
    # State Plane coordinates: same spatial information as Latitude/Longitude
    # but in a local projection that requires conversion for most analysis tools.
    "X Coordinate (State Plane)",
    "Y Coordinate (State Plane)",

    # Location: a "(lat, long)" string — fully redundant with the separate
    # Latitude and Longitude float columns.
    "Location",

    # Facility Type: 99.8 % missing in the 2026 sample; only 1 unique value
    # ("DSNY Garage") when populated. No analytical value.
    "Facility Type",

    # Due Date: 99.7 % missing. No analytical value.
    "Due Date",
]

df = df.drop(*UNCONDITIONAL_DROPS)

# ── Conditional drops — Bridge / Highway columns ─────────────────────────────
# These columns should only be populated for bridge/highway-related complaints.
# We check completeness *within* that subset before deciding to drop.

BRIDGE_COLS = ["Bridge Highway Name", "Bridge Highway Segment",
               "Bridge Highway Direction", "Road Ramp"]

bridge_mask = (
    F.lower(F.col("Complaint Type")).contains("bridge")
    | F.lower(F.col("Complaint Type")).contains("highway")
    | F.lower(F.col("Descriptor")).contains("bridge")
    | F.lower(F.col("Descriptor")).contains("highway")
)

n_bridge_rows = df.filter(bridge_mask).count()

if n_bridge_rows > 0:
    n_bridge_filled = (
        df.filter(bridge_mask)
          .filter(F.col("Bridge Highway Name").isNotNull())
          .count()
    )
    bridge_completeness = n_bridge_filled / n_bridge_rows
else:
    bridge_completeness = 0.0

if bridge_completeness < 0.20:
    # Below 20 % completeness even within relevant complaints → drop from Silver
    df = df.drop(*BRIDGE_COLS)
    print(f"[Step 4] Bridge/Highway columns dropped "
          f"(completeness within bridge complaints: {bridge_completeness:.1%})")
else:
    print(f"[Step 4] Bridge/Highway columns KEPT "
          f"(completeness within bridge complaints: {bridge_completeness:.1%})")

# ── Conditional drops — TLC-specific columns ─────────────────────────────────
TLC_COLS = ["Taxi Company Borough", "Taxi Pick Up Location", "Vehicle Type"]

tlc_mask = F.col("Agency") == "TLC"
n_tlc_rows = df.filter(tlc_mask).count()

if n_tlc_rows > 0:
    n_tlc_filled = (
        df.filter(tlc_mask)
          .filter(F.col("Taxi Company Borough").isNotNull())
          .count()
    )
    tlc_completeness = n_tlc_filled / n_tlc_rows
else:
    tlc_completeness = 0.0

if tlc_completeness < 0.20:
    df = df.drop(*TLC_COLS)
    print(f"[Step 4] TLC columns dropped "
          f"(completeness within TLC complaints: {tlc_completeness:.1%})")
else:
    print(f"[Step 4] TLC columns KEPT "
          f"(completeness within TLC complaints: {tlc_completeness:.1%})")

dq_metrics["step_04_column_drops"] = {
    "unconditional_drops":         UNCONDITIONAL_DROPS,
    "bridge_completeness_pct":     round(bridge_completeness * 100, 2),
    "bridge_cols_dropped":         bridge_completeness < 0.20,
    "tlc_completeness_pct":        round(tlc_completeness * 100, 2),
    "tlc_cols_dropped":            tlc_completeness < 0.20,
    "remaining_column_count":      len(df.columns),
    "note_park_borough":           "Park Borough retained until Step 11 Borough recovery"
}

print(f"[Step 4] Column pruning complete. Remaining columns: {len(df.columns)}")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 45, Finished, Available, Finished, False)

[Step 4] Bridge/Highway columns KEPT (completeness within bridge complaints: 99.5%)
[Step 4] TLC columns dropped (completeness within TLC complaints: 7.7%)
[Step 4] Column pruning complete. Remaining columns: 42


---
## Step 5 — Fix Impossible and Invalid Dates

Three date anomaly patterns are addressed here:

| Anomaly | Evidence | Fix |
|---|---|---|
| `Closed Date` with year 2025 when `Created Date` is 2026 | 2 rows; manual inspection confirms correct day/time, wrong year — agent typed 2025 instead of 2026 | Add 12 months with `F.add_months` |
| `Closed Date` < `Created Date` | Cannot be closed before opened; many share the `07:00:00` sentinel, suggesting a system default | Set `Closed Date = NULL` (complaint is valid; only the timestamp is wrong) |
| `Closed Date` ≈ `Created Date` (within 60 seconds) | Likely auto-generated test records or instant system closures | Quarantine for manual review |
| `Resolution Action Updated Date` < `Created Date` | ~88 k rows; inconsistent patterns across rows; cannot be reliably corrected | Keep as-is in Silver; document as known upstream data quality limitation; exclude from Gold |

> **`CLOSED_DATE_THRESHOLD_SECONDS`** — the "closed within N seconds of creation"
> threshold is a named constant so it is reproducible and easy to adjust.


In [34]:
CLOSED_DATE_THRESHOLD_SECONDS = 60   # rows closed within this many seconds → quarantine

# ── Fix 1: Closed Date year 2025 when Created Date is 2026 ───────────────────
# We identify rows where the year gap between Created and Closed is exactly -1
# (i.e. Closed is one year before Created). These are confirmed typos.

year_typo_mask = (
    F.col("Closed Date").isNotNull()
    & (F.year("Closed Date") == F.year("Created Date") - 1)
    & (F.month("Closed Date") == F.month("Created Date"))
    & (F.dayofmonth("Closed Date") == F.dayofmonth("Created Date"))
)

df = df.withColumn(
    "Closed Date",
    F.when(year_typo_mask, F.add_months(F.col("Closed Date"), 12))
     .otherwise(F.col("Closed Date"))
)

n_year_fixed = df.filter(year_typo_mask).count()  # count before fix for logging

# ── Fix 2: Closed Date < Created Date → NULL ──────────────────────────────────
# The complaint is legitimate (Status may be Closed and the resolution valid);
# only the closure timestamp is wrong, so we null it rather than dropping the row.

closed_before_created_mask = (
    F.col("Closed Date").isNotNull()
    & (F.col("Closed Date") < F.col("Created Date"))
)

df = df.withColumn(
    "Closed Date",
    F.when(closed_before_created_mask, F.lit(None).cast(T.TimestampType()))
     .otherwise(F.col("Closed Date"))
)

n_closed_before_created = df.filter(closed_before_created_mask).count()

# ── Fix 3: Closed Date within THRESHOLD seconds of Created Date → quarantine ──
# Flag with a dedicated column so the row is quarantined at write time.
# We do not drop these rows immediately — they still flow through downstream steps.

instant_close_mask = (
    F.col("Closed Date").isNotNull()
    & (
        F.col("Closed Date").cast("long") - F.col("Created Date").cast("long")
        <= CLOSED_DATE_THRESHOLD_SECONDS
    )
    & (F.col("Closed Date").cast("long") >= F.col("Created Date").cast("long"))
)

df = df.withColumn("_instant_close_flag", instant_close_mask)

n_instant_close = df.filter(F.col("_instant_close_flag")).count()

# ── Document: Resolution Action Updated Date < Created Date ──────────────────
# We do NOT attempt to fix these. The patterns are too inconsistent and varied
# to correct without access to NYC's internal system logs.
# The column stays in Silver for audit purposes only and will be excluded from Gold.

n_res_date_anomaly = (
    df.filter(
        F.col("Resolution Action Updated Date").isNotNull()
        & (F.col("Resolution Action Updated Date") < F.col("Created Date"))
    ).count()
)

dq_metrics["step_05_date_fixes"] = {
    "closed_date_year_typo_fixed":      n_year_fixed,
    "closed_before_created_nulled":     n_closed_before_created,
    "instant_close_flagged_quarantine": n_instant_close,
    "resolution_date_anomaly_documented": n_res_date_anomaly,
    "threshold_seconds":                CLOSED_DATE_THRESHOLD_SECONDS,
    "note": (
        "Resolution Action Updated Date anomalies documented as known upstream "
        "data quality limitation. Column kept in Silver for audit; excluded from Gold."
    )
}

print(f"[Step 5] Year typo (2025→2026) fixed: {n_year_fixed:,}")
print(f"[Step 5] Closed < Created (set to NULL): {n_closed_before_created:,}")
print(f"[Step 5] Instant-close flagged for quarantine: {n_instant_close:,}")
print(f"[Step 5] Resolution date anomaly (documented, not fixed): {n_res_date_anomaly:,}")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 46, Finished, Available, Finished, False)

[Step 5] Year typo (2025→2026) fixed: 0
[Step 5] Closed < Created (set to NULL): 0
[Step 5] Instant-close flagged for quarantine: 34,160
[Step 5] Resolution date anomaly (documented, not fixed): 592,338


---
## Step 6 — Fix Status / Closed Date Mismatches

The `Status` and `Closed Date` fields must not contradict each other.
Two mismatch patterns are present in the data:

| Mismatch | Interpretation | Fix |
|---|---|---|
| `Status = 'Open'` but `Closed Date` is populated | Agent closed the ticket but forgot to update Status | Set `Status = 'Closed'` |
| `Status = 'Closed'` but `Closed Date` is NULL | Ticket confirmed closed but no timestamp recorded | Use `Resolution Action Updated Date` as the best available proxy for `Closed Date`, and add a flag column `closed_date_imputed = TRUE` so downstream consumers know the value is inferred |

> The imputation flag is critical. `Resolution Action Updated Date` has known
> reliability issues (documented in Step 5). Using it as a proxy for a small
> number of rows is acceptable, but the imputed values must be distinguishable
> from real ones.

In [35]:
# ── Fix 1: Status = Open but Closed Date is present → update Status to Closed ──
open_with_close_date_mask = (
    (F.lower(F.trim(F.col("Status"))) == "open")
    & F.col("Closed Date").isNotNull()
)

n_open_with_close = df.filter(open_with_close_date_mask).count()

df = df.withColumn(
    "Status",
    F.when(open_with_close_date_mask, F.lit("Closed"))
     .otherwise(F.col("Status"))
)

# ── Fix 2: Status = Closed but Closed Date is NULL → impute from Resolution date ──
closed_no_date_mask = (
    (F.lower(F.trim(F.col("Status"))) == "closed")
    & F.col("Closed Date").isNull()
    & F.col("Resolution Action Updated Date").isNotNull()
)

n_closed_no_date = df.filter(closed_no_date_mask).count()

# Add the imputation flag column (FALSE for all rows by default, TRUE where we impute)
df = df.withColumn("closed_date_imputed", F.lit(False))

df = (
    df.withColumn(
        "Closed Date",
        F.when(closed_no_date_mask, F.col("Resolution Action Updated Date"))
         .otherwise(F.col("Closed Date"))
    )
    .withColumn(
        "closed_date_imputed",
        F.when(closed_no_date_mask, F.lit(True))
         .otherwise(F.col("closed_date_imputed"))
    )
)

dq_metrics["step_06_status_mismatch"] = {
    "status_corrected_open_to_closed":  n_open_with_close,
    "closed_date_imputed_from_resolution": n_closed_no_date,
    "note": (
        "closed_date_imputed=TRUE column added. Imputed values sourced from "
        "Resolution Action Updated Date which has known reliability issues (Step 5)."
    )
}

print(f"[Step 6] Status corrected (Open→Closed): {n_open_with_close:,}")
print(f"[Step 6] Closed Date imputed from Resolution Date: {n_closed_no_date:,}")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 47, Finished, Available, Finished, False)

[Step 6] Status corrected (Open→Closed): 6,967
[Step 6] Closed Date imputed from Resolution Date: 7


---
## Step 7 — Remove Internal Routing Codes

Two columns — `Problem Detail` and `Additional Details` — sometimes contain
short codes of the form `[A-Z]{2}[0-9]` (e.g. `NM1`, `WA2`). These are
internal NYC 311 routing codes that leaked into the public dataset.
Based on our data, this appears to be an **ongoing upstream pipeline defect**
(still present as of April 2026, not an isolated historical issue).

The rest of the complaint details in affected rows are valid, so **rows are
not dropped** — only the codes are removed.

| Column | Pattern behaviour | Action |
|---|---|---|
| `Additional Details` | The entire cell value is the routing code | Replace with `NULL` |
| `Problem Detail` | Routing code appended at the end of a valid description | Strip from the end of the string |

A count of affected rows is recorded per pipeline run. A **sudden spike or
disappearance** of these codes should be investigated — it may indicate the
upstream defect has been fixed, or has changed in form.

In [36]:
# ── Step 7: Remove Internal Routing Codes ─────────────────────────────────────
# Pattern: exactly 2 uppercase letters followed by exactly 1 digit
full_code_pattern    = r"^\s*[A-Z]{2}[0-9]\s*$"
trailing_code_pattern = r"\s+[A-Z]{2}[0-9]\s*$"

n_additional_details_nulled = 0
n_problem_detail_stripped   = 0

# ── Additional Details ────────────────────────────────────────────────────────
if "Additional Details" in df.columns:
    n_additional_details_nulled = (
        df.filter(F.col("Additional Details").isNotNull())
          .filter(F.col("Additional Details").rlike(full_code_pattern))
          .count()
    )
    df = df.withColumn(
        "Additional Details",
        F.when(
            F.col("Additional Details").isNotNull()
            & F.col("Additional Details").rlike(full_code_pattern),
            F.lit(None).cast(T.StringType())
        ).otherwise(F.col("Additional Details"))
    )
else:
    print("[Step 7] 'Additional Details' column not present in Bronze — skipping.")

# ── Problem Detail ────────────────────────────────────────────────────────────
if "Problem Detail" in df.columns:
    n_problem_detail_stripped = (
        df.filter(F.col("Problem Detail").isNotNull())
          .filter(F.col("Problem Detail").rlike(trailing_code_pattern))
          .count()
    )
    df = df.withColumn(
        "Problem Detail",
        F.when(
            F.col("Problem Detail").isNotNull(),
            F.regexp_replace(F.col("Problem Detail"), trailing_code_pattern, "")
        ).otherwise(F.col("Problem Detail"))
    )
else:
    print("[Step 7] 'Problem Detail' column not present in Bronze — skipping.")

dq_metrics["step_07_routing_codes"] = {
    "additional_details_nulled": n_additional_details_nulled,
    "problem_detail_stripped":   n_problem_detail_stripped,
    "total_affected_rows":       n_additional_details_nulled + n_problem_detail_stripped,
    "note": (
        "Columns absent from Bronze table if count is 0 and skipped messages appear. "
        "Monitor this count across runs — spike means defect worsening, "
        "zero means fixed upstream. Ongoing as of April 2026."
    )
}

print(f"[Step 7] Additional Details → NULL: {n_additional_details_nulled:,}")
print(f"[Step 7] Problem Detail trailing code stripped: {n_problem_detail_stripped:,}")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 48, Finished, Available, Finished, False)

[Step 7] 'Additional Details' column not present in Bronze — skipping.
[Step 7] 'Problem Detail' column not present in Bronze — skipping.
[Step 7] Additional Details → NULL: 0
[Step 7] Problem Detail trailing code stripped: 0


---
## Step 8 — Standardise Text Fields

For the Silver table to support reliable joins, grouping, and filtering, all
text columns with known categorical behaviour must have consistent casing and
value representations.

The following standardisations are applied:

| Column | Transformation | Reason |
|---|---|---|
| **All string columns** | `F.trim()` — strip leading/trailing whitespace | Silent killer for joins and GROUP BY; applied globally before any value mapping |
| `Street Name` | Upper Case → Title Case | Cosmetic consistency across agencies |
| `Borough` | Any case → Title Case | Consistency with Borough recovery in Step 11 |
| `Location Type` | 4 known non-standard values → official data dictionary values | Prevent duplicates in group-by |
| `Community Board` | Any value containing `"Unspecified"` → `"Unspecified"` | Borough info already in the Borough column |

In [37]:
# ── Global whitespace trim on all string columns ──────────────────────────────
# This is the single most impactful standardisation step for join correctness.
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, T.StringType)]

for col_name in string_cols:
    df = df.withColumn(col_name, F.trim(F.col(f"`{col_name}`")))

print(f"[Step 8] Trimmed whitespace on {len(string_cols)} string columns.")

# ── Street Name: UPPER CASE → Title Case ─────────────────────────────────────
df = df.withColumn(
    "Street Name",
    F.initcap(F.lower(F.col("Street Name")))
)

# ── Borough: any case → Title Case ───────────────────────────────────────────
df = df.withColumn(
    "Borough",
    F.initcap(F.lower(F.col("Borough")))
)

# ── Location Type: map non-standard values to the official data dictionary ────
location_type_mapping = {
    "COMERCIAL":                    "Commercial",
    "MIXED USE":                    "Mixed Use Building",
    "PARKING LOT OR GARAGE":        "Parking Lot/Garage",
    "FOOD ESTABLISHMENT OR VENDOR": "Restaurant/Bar/Deli/Bakery",
}

location_type_expr = F.col("Location Type")
for raw_val, clean_val in location_type_mapping.items():
    location_type_expr = F.when(
        F.upper(F.trim(F.col("Location Type"))) == raw_val,
        F.lit(clean_val)
    ).otherwise(location_type_expr)

df = df.withColumn("Location Type", location_type_expr)

# ── Community Board: collapse all "Unspecified" variants → "Unspecified" ─────
# Values like "Unspecified BROOKLYN", "0 Unspecified", "0 UNSPECIFIED BROOKLYN"
# all reduce to "Unspecified". Borough info in the Borough column is unaffected.
df = df.withColumn(
    "Community Board",
    F.when(
        F.lower(F.col("Community Board")).contains("unspecified"),
        F.lit("Unspecified")
    ).otherwise(F.col("Community Board"))
)

dq_metrics["step_08_text_standardisation"] = {
    "string_cols_trimmed":    len(string_cols),
    "location_type_remapped": len(location_type_mapping),
    "notes": "Borough Title Case applied here; further normalisation continues in Step 11."
}

print("[Step 8] Text standardisation complete.")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 49, Finished, Available, Finished, False)

[Step 8] Trimmed whitespace on 33 string columns.
[Step 8] Text standardisation complete.


---
## Step 9 — Business Rule Validation

We validate categorical and range-bounded columns against known business rules.
Rows that violate a rule are **quarantined** (not dropped) because a violation
could represent either:
- a genuine data entry error that should be corrected, or
- a new valid value that has not yet been added to our domain list (e.g. a new NYC agency).

| Field | Rule | Quarantine tag |
|---|---|---|
| `Agency` | Must be one of the 15 known NYC agency codes | `INVALID_AGENCY` |
| `Status` | Must be one of 7 known status values | `INVALID_STATUS` |
| `Council District` | Must be 1–51 (NYC has 51 districts) | `COUNCIL_DISTRICT_OUT_OF_RANGE` |
| `Police Precinct` | Must be 1–123 or `"Unspecified"` | `POLICE_PRECINCT_OUT_OF_RANGE` |
| `Incident Zip` | Must be a 5-character all-digit string | `INVALID_ZIP_FORMAT` |

In [38]:
# Helper: build a quarantine mask and tag matching rows, then remove them from df
def quarantine_violations(df, mask, reason_tag):
    df_q = tag_quarantine(df.filter(mask), reason_tag)
    df_c = df.filter(~mask)
    return df_c, df_q

print(f"Rows entering Step 9: {df.count():,}")

# ── Agency validation ─────────────────────────────────────────────────────────
agency_invalid_mask = (
    F.col("Agency").isNotNull()
    & ~F.col("Agency").isin(list(VALID_AGENCIES))
)
df, df_q_agency = quarantine_violations(df, agency_invalid_mask, "INVALID_AGENCY")
quarantine_frames.append((df_q_agency, "INVALID_AGENCY"))
print(f"After Agency check:    {df.count():,}  (quarantined: {df_q_agency.count():,})")

# ── Status validation ─────────────────────────────────────────────────────────
status_invalid_mask = (
    F.col("Status").isNotNull()
    & ~F.col("Status").isin(list(VALID_STATUSES))
)
df, df_q_status = quarantine_violations(df, status_invalid_mask, "INVALID_STATUS")
quarantine_frames.append((df_q_status, "INVALID_STATUS"))
print(f"After Status check:    {df.count():,}  (quarantined: {df_q_status.count():,})")

# ── Council District range check ──────────────────────────────────────────────
council_invalid_mask = (
    F.col("Council District").isNotNull()
    & (
        (F.col("Council District") < 1)
        | (F.col("Council District") > 51)
    )
)
df, df_q_council = quarantine_violations(df, council_invalid_mask, "COUNCIL_DISTRICT_OUT_OF_RANGE")
quarantine_frames.append((df_q_council, "COUNCIL_DISTRICT_OUT_OF_RANGE"))
print(f"After Council check:   {df.count():,}  (quarantined: {df_q_council.count():,})")

# ── Police Precinct range check ───────────────────────────────────────────────
# Use rlike to check numeric-only values instead of casting to int,
# which returns NULL for non-numeric strings and causes mask misbehaviour.
precinct_invalid_mask = (
    F.col("Police Precinct").isNotNull()
    & (F.lower(F.col("Police Precinct")) != "unspecified")
    & F.col("Police Precinct").rlike(r"^\d+$")   # only check rows that are purely numeric
    & (
        (F.col("Police Precinct").cast(T.IntegerType()) < 1)
        | (F.col("Police Precinct").cast(T.IntegerType()) > 123)
    )
)
df, df_q_precinct = quarantine_violations(df, precinct_invalid_mask, "POLICE_PRECINCT_OUT_OF_RANGE")
quarantine_frames.append((df_q_precinct, "POLICE_PRECINCT_OUT_OF_RANGE"))
print(f"After Precinct check:  {df.count():,}  (quarantined: {df_q_precinct.count():,})")

# ── Incident Zip format check ─────────────────────────────────────────────────
# Only quarantine ZIPs that are non-null AND non-empty AND not 5 digits.
# "00000" is technically valid after padding — allow it through.
zip_invalid_mask = (
    F.col("Incident Zip").isNotNull()
    & (F.trim(F.col("Incident Zip")) != "")
    & ~F.col("Incident Zip").rlike(r"^\d{5}$")
)
df, df_q_zip = quarantine_violations(df, zip_invalid_mask, "INVALID_ZIP_FORMAT")
quarantine_frames.append((df_q_zip, "INVALID_ZIP_FORMAT"))
print(f"After ZIP check:       {df.count():,}  (quarantined: {df_q_zip.count():,})")

# ── Cross-validate Incident Zip against Borough (flag only, no quarantine) ────
non_nyc_zip_mask = (
    F.col("Incident Zip").isNotNull()
    & F.col("Borough").isNotNull()
    & (F.lower(F.col("Borough")) != "unspecified")
    & ~F.col("Incident Zip").rlike(r"^1[01]")
)
n_non_nyc_zip = df.filter(non_nyc_zip_mask).count()
df = df.withColumn("_non_nyc_zip_flag", non_nyc_zip_mask)
print(f"Non-NYC ZIP flagged (not quarantined): {n_non_nyc_zip:,}")

dq_metrics["step_09_business_rules"] = {
    "invalid_agency_quarantined":          df_q_agency.count(),
    "invalid_status_quarantined":          df_q_status.count(),
    "council_district_oob_quarantined":    df_q_council.count(),
    "police_precinct_oob_quarantined":     df_q_precinct.count(),
    "invalid_zip_format_quarantined":      df_q_zip.count(),
    "non_nyc_zip_flagged_not_quarantined": n_non_nyc_zip,
    "rows_remaining":                      df.count(),
}

print(f"\n[Step 9] Rows remaining: {df.count():,}")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 50, Finished, Available, Finished, False)

Rows entering Step 9: 1,207,334
After Agency check:    1,200,676  (quarantined: 6,658)
After Status check:    1,200,676  (quarantined: 0)
After Council check:   1,200,676  (quarantined: 0)
After Precinct check:  1,200,676  (quarantined: 0)
After ZIP check:       1,200,676  (quarantined: 0)
Non-NYC ZIP flagged (not quarantined): 9

[Step 9] Rows remaining: 1,200,676


---
## Step 10 — Validate Coordinates

Coordinate validation happens on two tiers:

| Tier | Rule | Action |
|---|---|---|
| **Global** | `Latitude` ∈ [−90, 90]; `Longitude` ∈ [−180, 180] | Quarantine row |
| **NYC-local** | When `City` is NYC or `Borough` is known: `Latitude` ∈ [40.4774, 40.9176]; `Longitude` ∈ [−74.2591, −73.7004] | Quarantine row |

If coordinates are globally/locally invalid they are **set to NULL** for rows
where the Borough is already known (the row is still usable for non-spatial
analysis). Rows where coordinates are the *only* spatial information and they
are invalid go to quarantine.

Additionally, we enforce **coordinate pair completeness**: a row must have both
Latitude and Longitude, or neither. A row with only one is ambiguous.

In [39]:
# ── Bounding boxes ────────────────────────────────────────────────────────────
LAT_GLOBAL_MIN, LAT_GLOBAL_MAX = -90.0,   90.0
LON_GLOBAL_MIN, LON_GLOBAL_MAX = -180.0,  180.0

LAT_NYC_MIN, LAT_NYC_MAX = 40.4774, 40.9176
LON_NYC_MIN, LON_NYC_MAX = -74.2591, -73.7004

nyc_location_mask = (
    (F.lower(F.trim(F.col("City"))) == "new york")
    | (
        F.col("Borough").isNotNull()
        & (F.lower(F.col("Borough")) != "unspecified")
    )
)

# ── Global range violations ───────────────────────────────────────────────────
global_coord_invalid_mask = (
    F.col("Latitude").isNotNull()
    & (
        (F.col("Latitude")  < LAT_GLOBAL_MIN) | (F.col("Latitude")  > LAT_GLOBAL_MAX)
        | (F.col("Longitude") < LON_GLOBAL_MIN) | (F.col("Longitude") > LON_GLOBAL_MAX)
    )
)

# Set invalid coordinates to NULL rather than quarantining the row
# (the complaint itself is still valid)
df = df.withColumn(
    "Latitude",
    F.when(global_coord_invalid_mask, F.lit(None).cast(T.DoubleType()))
     .otherwise(F.col("Latitude"))
)
df = df.withColumn(
    "Longitude",
    F.when(global_coord_invalid_mask, F.lit(None).cast(T.DoubleType()))
     .otherwise(F.col("Longitude"))
)

n_global_invalid = df.filter(global_coord_invalid_mask).count()

# ── NYC-local range violations ────────────────────────────────────────────────
local_coord_invalid_mask = (
    nyc_location_mask
    & F.col("Latitude").isNotNull()
    & (
        (F.col("Latitude")  < LAT_NYC_MIN) | (F.col("Latitude")  > LAT_NYC_MAX)
        | (F.col("Longitude") < LON_NYC_MIN) | (F.col("Longitude") > LON_NYC_MAX)
    )
)

df, df_q_local_coords = quarantine_violations(
    df, local_coord_invalid_mask, "COORDINATES_OUTSIDE_NYC_BOUNDS"
)
quarantine_frames.append((df_q_local_coords, "COORDINATES_OUTSIDE_NYC_BOUNDS"))

# ── Coordinate pair completeness ──────────────────────────────────────────────
# Exactly one of (Latitude, Longitude) is non-NULL → ambiguous record
one_coord_only_mask = (
    (F.col("Latitude").isNull()  & F.col("Longitude").isNotNull())
    | (F.col("Latitude").isNotNull() & F.col("Longitude").isNull())
)

df, df_q_one_coord = quarantine_violations(
    df, one_coord_only_mask, "SINGLE_COORDINATE_ONLY"
)
quarantine_frames.append((df_q_one_coord, "SINGLE_COORDINATE_ONLY"))

# ── Cross-validate Borough vs coordinates ─────────────────────────────────────
# Flag rows where Borough is assigned but coordinates fall outside the expected
# borough bounding box. This catches a different class of error from the pure
# range check above (e.g. Borough=Brooklyn but coordinates in the Bronx).
# We flag rather than quarantine because the Borough may be wrong, not the coords.
# Full cross-validation requires the GeoJSON shapes loaded in Step 11.
#
# For now we record the count for the DQ metrics; Step 11 spatial join will
# naturally expose these mismatches when the recovered borough differs from the
# stored one.

dq_metrics["step_10_coordinate_validation"] = {
    "global_range_invalid_coords_nulled":    n_global_invalid,
    "local_nyc_range_quarantined":           df_q_local_coords.count(),
    "single_coordinate_quarantined":         df_q_one_coord.count(),
    "rows_remaining":                        df.count()
}

print(f"[Step 10] Global coord violations (nulled): {n_global_invalid:,}")
print(f"[Step 10] Outside NYC bounds (quarantined): {df_q_local_coords.count():,}")
print(f"[Step 10] Single coordinate only (quarantined): {df_q_one_coord.count():,}")
print(f"[Step 10] Rows remaining: {df.count():,}")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 51, Finished, Available, Finished, False)

[Step 10] Global coord violations (nulled): 0
[Step 10] Outside NYC bounds (quarantined): 13
[Step 10] Single coordinate only (quarantined): 0
[Step 10] Rows remaining: 1,200,663


---
## Step 11 — Recover Borough from Unspecified / NULL Rows

`Borough` is one half of the composite join key used in the Gold layer, so rows
with `Borough = NULL` or `Borough = "Unspecified"` cannot be included in Gold
without recovery.

A **four-tier recovery cascade** is applied, from cheapest to most expensive:

| Tier | Source | Method |
|---|---|---|
| 1 | `Park Borough` | Direct copy if not NULL |
| 2 | `BBL` first digit | Maps `1→Manhattan`, `2→Bronx`, `3→Brooklyn`, `4→Queens`, `5→Staten Island` |
| 3 | `City → Borough` dictionary | Built from rows in *this* dataset that already have a Borough; used to recover rows that share the same City |
| 4 | Geospatial lookup | Load NYC Borough Boundaries GeoJSON; find which polygon contains each complaint's coordinates using `geopandas` + `shapely` |

After all four tiers, rows still `Unspecified` go to the Silver quarantine table.
Some of these will have `City` values outside NYC (Las Vegas, San Diego, etc.),
which is why no borough can be assigned.

> **Performance note:** the geospatial join (Tier 4) runs as a vectorised pandas
> UDF via `mapInPandas` to avoid a row-by-row Python loop across the Spark cluster.


In [40]:
# We work on the full df (which includes the _pending_borough_recovery flag)
unspecified_mask = (
    F.col("_pending_borough_recovery")
    | F.col("Borough").isNull()
    | (F.lower(F.trim(F.col("Borough"))) == "unspecified")
)

# ── Tier 1: Park Borough ──────────────────────────────────────────────────────
df = df.withColumn(
    "Borough",
    F.when(
        unspecified_mask & F.col("Park Borough").isNotNull(),
        F.initcap(F.lower(F.col("Park Borough")))
    ).otherwise(F.col("Borough"))
)

unspecified_mask = (
    F.col("Borough").isNull()
    | (F.lower(F.trim(F.col("Borough"))) == "unspecified")
)

n_recovered_tier1 = (
    df.filter(F.col("_pending_borough_recovery"))
      .filter(~(F.col("Borough").isNull() | (F.lower(F.col("Borough")) == "unspecified")))
      .count()
)

# ── Tier 2: BBL first digit ───────────────────────────────────────────────────
bbl_borough_expr = F.col("Borough")
for digit, borough_name in BOROUGH_BBL_MAP.items():
    bbl_borough_expr = F.when(
        unspecified_mask
        & F.col("BBL").isNotNull()
        & (F.substring(F.col("BBL"), 1, 1) == digit),
        F.lit(borough_name)
    ).otherwise(bbl_borough_expr)

df = df.withColumn("Borough", bbl_borough_expr)
unspecified_mask = F.col("Borough").isNull() | (F.lower(F.trim(F.col("Borough"))) == "unspecified")

n_recovered_tier2 = (
    df.filter(F.col("_pending_borough_recovery"))
      .filter(~unspecified_mask)
      .count() - n_recovered_tier1
)

# ── Tier 3: City → Borough mapping ───────────────────────────────────────────
city_borough_map_df = (
    df.filter(~unspecified_mask)
      .filter(F.col("City").isNotNull())
      .groupBy("City", "Borough")
      .count()
      .withColumn("_rank",
                  F.row_number().over(
                      Window.partitionBy("City").orderBy(F.col("count").desc())
                  ))
      .filter(F.col("_rank") == 1)
      .drop("count", "_rank")
)

city_borough_map = city_borough_map_df.toPandas().set_index("City")["Borough"].to_dict()

city_borough_expr = F.col("Borough")
for city, borough in city_borough_map.items():
    city_borough_expr = F.when(
        unspecified_mask & (F.trim(F.col("City")) == city),
        F.lit(borough)
    ).otherwise(city_borough_expr)

df = df.withColumn("Borough", city_borough_expr)
unspecified_mask = F.col("Borough").isNull() | (F.lower(F.trim(F.col("Borough"))) == "unspecified")

n_recovered_tier3 = (
    df.filter(F.col("_pending_borough_recovery"))
      .filter(~unspecified_mask)
      .count() - n_recovered_tier1 - n_recovered_tier2
)

# ── Tier 4: Geo-lookup — DISABLED (OOM risk on 1.2M rows via mapInPandas) ────
# Upload borough_boundaries.geojson to Files/project_data/reference/ and run
# geo-recovery as a separate offline job against quarantine rows only.
n_for_geo_recovery = df.filter(unspecified_mask & F.col("Latitude").isNotNull()).count()
print(
    f"[Step 11 Tier 4] SKIPPED — geo-lookup disabled to prevent OOM. "
    f"{n_for_geo_recovery:,} coordinate-eligible rows will go to quarantine. "
    f"Run geo-recovery as a separate job against the quarantine table."
)

# ── Quarantine rows still unspecified after all tiers ────────────────────────
df_q_unspecified_borough = tag_quarantine(
    df.filter(unspecified_mask), "UNRECOVERED_BOROUGH"
)
quarantine_frames.append((df_q_unspecified_borough, "UNRECOVERED_BOROUGH"))
df = df.filter(~unspecified_mask).drop("_pending_borough_recovery")

n_still_unrecovered = df_q_unspecified_borough.count()

dq_metrics["step_11_borough_recovery"] = {
    "recovered_tier1_park_borough": n_recovered_tier1,
    "recovered_tier2_bbl":          n_recovered_tier2,
    "recovered_tier3_city_map":     n_recovered_tier3,
    "tier4_geo_lookup":             "DISABLED — OOM risk",
    "still_unrecovered_quarantined": n_still_unrecovered,
}

print(f"[Step 11] Tier 1 (Park Borough): {n_recovered_tier1:,}")
print(f"[Step 11] Tier 2 (BBL):          {n_recovered_tier2:,}")
print(f"[Step 11] Tier 3 (City map):     {n_recovered_tier3:,}")
print(f"[Step 11] Quarantined (unrecovered): {n_still_unrecovered:,}")
print(f"[Step 11] Rows remaining in df:  {df.count():,}")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 52, Finished, Available, Finished, False)

[Step 11 Tier 4] SKIPPED — geo-lookup disabled to prevent OOM. 46 coordinate-eligible rows will go to quarantine. Run geo-recovery as a separate job against the quarantine table.
[Step 11] Tier 1 (Park Borough): 0
[Step 11] Tier 2 (BBL):          3
[Step 11] Tier 3 (City map):     43
[Step 11] Quarantined (unrecovered): 1,022
[Step 11] Rows remaining in df:  1,199,641


---
## Step 12 — Recover Missing Park Facility Names

`Park Facility Name` should only be populated for DPR (Parks Department)
complaints. A missing value for a non-DPR complaint is **expected** and is
not an error. We only attempt recovery for DPR rows where the name is absent.

**Recovery strategy (two tiers):**
1. Coordinate dictionary: `{(latitude, longitude): Park Facility Name}` built
   from DPR rows in this dataset that already have a name. Coordinates are
   rounded to 4 decimal places (≈ 11 m) to handle minor floating-point variation.
2. Geospatial lookup: load the NYC Parks Properties GeoJSON; find which park
   polygon each complaint's coordinates fall inside.

In [41]:
dpr_missing_park_mask = (
    (F.col("Agency") == "DPR")
    & (
        F.col("Park Facility Name").isNull()
        | (F.lower(F.trim(F.col("Park Facility Name"))) == "unspecified")
    )
)

n_dpr_missing = df.filter(dpr_missing_park_mask).count()
print(f"[Step 12] DPR rows missing Park Facility Name: {n_dpr_missing:,}")

n_recovered_park_tier1 = 0
n_recovered_park_tier2 = 0

if n_dpr_missing > 0:
    # ── Tier 1: Coordinate dictionary ────────────────────────────────────────
    coord_park_map = (
        df.filter(F.col("Agency") == "DPR")
          .filter(F.col("Park Facility Name").isNotNull())
          .filter(F.lower(F.trim(F.col("Park Facility Name"))) != "unspecified")
          .withColumn("_lat_r", F.round(F.col("Latitude"),  4))
          .withColumn("_lon_r", F.round(F.col("Longitude"), 4))
          .groupBy("_lat_r", "_lon_r")
          .agg(F.first("Park Facility Name").alias("park_name"))
          .toPandas()
          .set_index(["_lat_r", "_lon_r"])["park_name"]
          .to_dict()
    )

    lookup_park_udf = F.udf(
        lambda lat, lon, name: (
            coord_park_map.get((round(lat, 4), round(lon, 4)))
            if lat and lon and (not name or name.lower().strip() == "unspecified")
            else name
        ),
        T.StringType()
    )

    df = df.withColumn(
        "Park Facility Name",
        F.when(
            dpr_missing_park_mask & F.col("Latitude").isNotNull(),
            lookup_park_udf(F.col("Latitude"), F.col("Longitude"), F.col("Park Facility Name"))
        ).otherwise(F.col("Park Facility Name"))
    )

    n_recovered_park_tier1 = n_dpr_missing - df.filter(dpr_missing_park_mask).count()
    print(f"[Step 12 Tier 1] Recovered via coordinate dict: {n_recovered_park_tier1:,}")

    # ── Tier 2: Geo-lookup — DISABLED (OOM risk) ──────────────────────────────
    n_still_missing_park = df.filter(dpr_missing_park_mask).count()
    print(
        f"[Step 12 Tier 2] SKIPPED — geo-lookup disabled to prevent OOM. "
        f"{n_still_missing_park:,} DPR rows remain without a Park Facility Name."
    )

dq_metrics["step_12_park_facility_recovery"] = {
    "dpr_rows_missing_park_name":    n_dpr_missing,
    "recovered_tier1_coord_dict":    n_recovered_park_tier1,
    "tier2_geo_lookup":              "DISABLED — OOM risk",
    "still_unrecovered":             n_dpr_missing - n_recovered_park_tier1,
}

print(f"[Step 12] Complete.")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 53, Finished, Available, Finished, False)

[Step 12] DPR rows missing Park Facility Name: 22,372
[Step 12 Tier 1] Recovered via coordinate dict: 195
[Step 12 Tier 2] SKIPPED — geo-lookup disabled to prevent OOM. 22,177 DPR rows remain without a Park Facility Name.
[Step 12] Complete.


---
## Step 13 — Recover Missing Council Districts

The same two-tier recovery strategy from Step 12 is applied here.
Council district boundaries are loaded from the NYC City Council Districts
GeoJSON file.

> **Versioning note:** Council district boundaries change after each decennial
> redistricting cycle. The GeoJSON file at `COUNCIL_GEOJSON_PATH` should be
> pinned to a specific version and documented with the redistricting year it
> reflects. Using the wrong boundary year will silently assign incorrect districts.

In [42]:
council_missing_mask = F.col("Council District").isNull()
n_council_missing = df.filter(council_missing_mask).count()
print(f"[Step 13] Rows missing Council District: {n_council_missing:,}")

n_recovered_council_tier1 = 0

if n_council_missing > 0:
    # ── Tier 1: Coordinate dictionary ────────────────────────────────────────
    coord_council_map = (
        df.filter(F.col("Council District").isNotNull())
          .filter(F.col("Latitude").isNotNull())
          .withColumn("_lat_r", F.round(F.col("Latitude"),  4))
          .withColumn("_lon_r", F.round(F.col("Longitude"), 4))
          .groupBy("_lat_r", "_lon_r")
          .agg(F.first("Council District").alias("council_dist"))
          .toPandas()
          .set_index(["_lat_r", "_lon_r"])["council_dist"]
          .to_dict()
    )

    lookup_council_udf = F.udf(
        lambda lat, lon: coord_council_map.get(
            (round(lat, 4), round(lon, 4)) if lat and lon else None
        ),
        T.IntegerType()
    )

    df = df.withColumn(
        "Council District",
        F.when(
            council_missing_mask & F.col("Latitude").isNotNull(),
            lookup_council_udf(F.col("Latitude"), F.col("Longitude"))
        ).otherwise(F.col("Council District"))
    )

    n_recovered_council_tier1 = n_council_missing - df.filter(council_missing_mask).count()
    print(f"[Step 13 Tier 1] Recovered via coordinate dict: {n_recovered_council_tier1:,}")

    # ── Tier 2: Geo-lookup — DISABLED (OOM risk) ──────────────────────────────
    n_still_missing_council = df.filter(council_missing_mask).count()
    print(
        f"[Step 13 Tier 2] SKIPPED — geo-lookup disabled to prevent OOM. "
        f"{n_still_missing_council:,} rows remain without a Council District."
    )

n_still_missing_council = df.filter(council_missing_mask).count()

dq_metrics["step_13_council_district_recovery"] = {
    "rows_missing_council_district": n_council_missing,
    "recovered_tier1_coord_dict":    n_recovered_council_tier1,
    "tier2_geo_lookup":              "DISABLED — OOM risk",
    "still_unrecovered":             n_still_missing_council,
    "versioning_note": (
        "Council district boundaries change after each redistricting cycle. "
        "Pin the GeoJSON to a specific redistricting year when geo-lookup is re-enabled."
    )
}

print(f"[Step 13] Complete. Still missing Council District: {n_still_missing_council:,}")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 54, Finished, Available, Finished, False)

[Step 13] Rows missing Council District: 38,722
[Step 13 Tier 1] Recovered via coordinate dict: 1,766
[Step 13 Tier 2] SKIPPED — geo-lookup disabled to prevent OOM. 36,956 rows remain without a Council District.
[Step 13] Complete. Still missing Council District: 36,956


---
## Step 14 — Handle BBL Nulls and Invalid Values

BBL (Borough-Block-Lot) is a 10-digit property identifier. Not every complaint
type is associated with a specific building, so NULL BBL is **expected** in
many cases.

| Scenario | Interpretation | Action |
|---|---|---|
| `BBL = "0.0"` (→ `"0"` after Step 3) | Borough code 0 is invalid; NYC boroughs are 1–5 | Set to NULL |
| `BBL = NULL` + `Address Type ∈ {INTERSECTION, BLOCKFACE, PLACE, PLACENAME}` | Complaint references a street corner/area, not a building | NULL is correct — no action |
| `BBL = NULL` + `Address Type = ADDRESS` | Complaint linked to a specific building with an address | True missing value; cannot reliably fill — document as data quality limitation |

In [43]:
# ── Fix: BBL = "0" is not a valid property identifier ────────────────────────
# After Step 3, "0.0" has been cast to "0". Set these to NULL.
df = df.withColumn(
    "BBL",
    F.when(F.col("BBL") == "0", F.lit(None).cast(T.StringType()))
     .otherwise(F.col("BBL"))
)

n_bbl_zero_nulled = df.filter(F.col("BBL").isNull() & (F.col("BBL") == "0")).count()

# ── Document: expected NULLs for non-building complaints ──────────────────────
INTERSECTION_ADDRESS_TYPES = {"INTERSECTION", "BLOCKFACE", "PLACE", "PLACENAME"}

n_bbl_null_expected = (
    df.filter(F.col("BBL").isNull())
      .filter(F.upper(F.trim(F.col("Address Type"))).isin(list(INTERSECTION_ADDRESS_TYPES)))
      .count()
)

# ── Document: unexpected NULLs for address-type complaints ────────────────────
n_bbl_null_address = (
    df.filter(F.col("BBL").isNull())
      .filter(F.upper(F.trim(F.col("Address Type"))) == "ADDRESS")
      .count()
)

dq_metrics["step_14_bbl"] = {
    "bbl_zero_set_to_null":             n_bbl_zero_nulled,
    "bbl_null_expected_non_building":   n_bbl_null_expected,
    "bbl_null_unexpected_address_type": n_bbl_null_address,
    "notes": (
        "Unexpected ADDRESS-type BBL NULLs documented as data quality limitation. "
        "Future improvement: cross-reference NYC PLUTO dataset for address→BBL lookup."
    )
}

print(f"[Step 14] BBL='0' set to NULL: (already corrected above)")
print(f"[Step 14] BBL NULL — expected (intersection/area): {n_bbl_null_expected:,}")
print(f"[Step 14] BBL NULL — unexpected (address type): {n_bbl_null_address:,} (documented, not fixed)")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 55, Finished, Available, Finished, False)

[Step 14] BBL='0' set to NULL: (already corrected above)
[Step 14] BBL NULL — expected (intersection/area): 120,189
[Step 14] BBL NULL — unexpected (address type): 27,729 (documented, not fixed)


---
## Step 15 — Capture Status Changes Over Time (Append-Based SCD Tracking)

The pipeline runs daily. Each day, the last 7 days of complaints are pulled
from NYC Open Data and processed through this notebook. Because `Status` changes
over the lifecycle of a complaint (`Open → In Progress → Closed`), we need
to **preserve the history** — not overwrite it.

**Design: append-based status log**

A `change_date` column (the pipeline execution date) is added to each row.
New complaints → appended with `change_date = today`.  
Existing complaints with unchanged status → skipped.  
Existing complaints with a new status → appended as a new row with the updated status.

| `unique_key` | `status` | `change_date` |
|---|---|---|
| ADX-123 | Open | 2026-02-10 |
| ADX-123 | In Progress | 2026-02-12 |
| ADX-123 | Closed | 2026-02-15 |

*To get the current status of any complaint: `SELECT … ORDER BY change_date DESC LIMIT 1`.*  
*To measure time in each status: subtract consecutive `change_date` values.*

**Known limitation:** The 7-day window is anchored to `Created Date`. A complaint
created more than 7 days ago that receives a status update will **not** be
captured by this pipeline run. Based on the 2026 sample data, the median
resolution time is within 7 days, so the majority of status changes are captured.
Long-running complaints (housing, infrastructure) are documented as a known gap.


In [45]:
# Add change_date to every row in the current cleaned batch
df = df.withColumn("change_date", F.lit(TODAY).cast(T.DateType()))

# ── Check if Silver table already exists ──────────────────────────────────────
try:
    spark.table(SILVER_TABLE).limit(1).collect()
    silver_table_exists = True
except Exception:
    silver_table_exists = False

if not silver_table_exists:
    print("[Step 15] Silver table not found — will write initial full load.")
    IS_INITIAL_LOAD = True

else:
    IS_INITIAL_LOAD = False
    df_silver = spark.table(SILVER_TABLE)

    # Silver is stored in snake_case — use snake_case to read it
    window_latest = Window.partitionBy("unique_key").orderBy(F.col("change_date").desc())
    df_silver_latest = (
        df_silver
        .withColumn("_rank", F.row_number().over(window_latest))
        .filter(F.col("_rank") == 1)
        .drop("_rank")
        .select("unique_key", "status")
        .withColumnRenamed("unique_key", "_silver_unique_key")
        .withColumnRenamed("status", "_silver_status")
    )

    # df still has Title Case at this point — join on Title Case side
    # using a rename to match the snake_case Silver key
    df_with_silver = df.join(
        df_silver_latest,
        df["Unique Key"] == df_silver_latest["_silver_unique_key"],
        how="left"
    ).drop("_silver_unique_key")

    df_new = df_with_silver.filter(F.col("_silver_status").isNull()).drop("_silver_status")
    df_updated = df_with_silver.filter(
        F.col("_silver_status").isNotNull()
        & (F.col("Status") != F.col("_silver_status"))
    ).drop("_silver_status")

    n_new     = df_new.count()
    n_updated = df_updated.count()
    n_skipped = df.count() - n_new - n_updated

    df = df_new.union(df_updated)

    dq_metrics["step_15_scd_tracking"] = {
        "new_complaints_appended":   n_new,
        "status_updates_appended":   n_updated,
        "unchanged_records_skipped": n_skipped,
        "known_limitation": (
            "7-day window anchored to Created Date. Complaints older than 7 days "
            "that receive a status update are not captured in this run."
        )
    }

    print(f"[Step 15] New: {n_new:,} | Updated: {n_updated:,} | Skipped: {n_skipped:,}")

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 57, Finished, Available, Finished, False)

[Step 15] New: 1,179,115 | Updated: 0 | Skipped: 20,526


---
## Step 16 — Write to Silver as a Delta Table

After all cleaning and SCD logic, four things happen atomically:

1. **Parquet files** are written to the `silver_stg_nyc_311` folder on OneLake,
   partitioned by day of `Created Date`. With ~11,350 new requests/day and a
   ~3× SCD multiplier, each run adds roughly 30,000–50,000 rows.
2. **Delta log entry** is created — records schema, files added, timestamp,
   and row count. Full audit trail; every write is permanently recorded.
3. **Schema enforcement** — any future write that violates the Silver schema
   (wrong type, NULL in a NOT NULL column, unexpected column) is rejected by
   Delta before touching any Parquet file.
4. **Table becomes queryable** — the SQL Analytics Endpoint syncs within ~60s;
   `silver_stg_nyc_311` is immediately available for T-SQL and Gold notebook joins.

Quarantined rows (from all steps) are flushed to `silver_quarantine_311` in
the same write cycle so the quarantine table is always in sync with the Silver table.


In [46]:
# ── Route instant-close quarantine rows (flagged in Step 5) ──────────────────
df_instant_close_quarantine = (
    df.filter(F.col("_instant_close_flag"))
      .withColumn("quarantine_reason", F.lit("INSTANT_CLOSE_WITHIN_60S"))
      .withColumn("quarantine_date",   F.lit(TODAY))
)
quarantine_frames.append((df_instant_close_quarantine, "INSTANT_CLOSE_WITHIN_60S"))
df = df.filter(~F.col("_instant_close_flag")).drop("_instant_close_flag")

# Drop internal helper columns
INTERNAL_COLS = ["_non_nyc_zip_flag", "_pending_borough_recovery"]
for col in INTERNAL_COLS:
    if col in df.columns:
        df = df.drop(col)

# ── Rename Title Case columns back to snake_case for Delta compatibility ──────
# Delta does not allow spaces or special characters in column names.
WRITE_RENAME_MAP = {
    "Unique Key":                       "unique_key",
    "Created Date":                     "created_date",
    "Closed Date":                      "closed_date",
    "Agency":                           "agency",
    "Agency Name":                      "agency_name",
    "Complaint Type":                   "complaint_type",
    "Descriptor":                       "descriptor",
    "Location Type":                    "location_type",
    "Incident Zip":                     "incident_zip",
    "Incident Address":                 "incident_address",
    "Street Name":                      "street_name",
    "Cross Street 1":                   "cross_street_1",
    "Cross Street 2":                   "cross_street_2",
    "Intersection Street 1":            "intersection_street_1",
    "Intersection Street 2":            "intersection_street_2",
    "Address Type":                     "address_type",
    "City":                             "city",
    "Landmark":                         "landmark",
    "Facility Type":                    "facility_type",
    "Status":                           "status",
    "Community Board":                  "community_board",
    "BBL":                              "bbl",
    "Borough":                          "borough",
    "X Coordinate (State Plane)":       "x_coordinate_state_plane",
    "Y Coordinate (State Plane)":       "y_coordinate_state_plane",
    "Park Facility Name":               "park_facility_name",
    "Park Borough":                     "park_borough",
    "Latitude":                         "latitude",
    "Longitude":                        "longitude",
    "Location":                         "location",
    "Due Date":                         "due_date",
    "Resolution Description":           "resolution_description",
    "Resolution Action Updated Date":   "resolution_action_updated_date",
    "Taxi Company Borough":             "taxi_company_borough",
    "Taxi Pick Up Location":            "taxi_pick_up_location",
    "Bridge Highway Name":              "bridge_highway_name",
    "Bridge Highway Direction":         "bridge_highway_direction",
    "Road Ramp":                        "road_ramp",
    "Bridge Highway Segment":           "bridge_highway_segment",
    "Vehicle Type":                     "vehicle_type",
    "Police Precinct":                  "police_precinct",
    "Council District":                 "council_district",
    "Additional Details":               "additional_details",
    "Problem Detail":                   "problem_detail",
}

for old, new in WRITE_RENAME_MAP.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)

# Apply same rename to all quarantine frames before writing
def rename_for_write(frame_df):
    for old, new in WRITE_RENAME_MAP.items():
        if old in frame_df.columns:
            frame_df = frame_df.withColumnRenamed(old, new)
    return frame_df

# ── Write clean rows to Silver ────────────────────────────────────────────────
rows_to_write = df.count()

if IS_INITIAL_LOAD:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("ingest_date")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"[Step 16] Initial load: {rows_to_write:,} rows written to {SILVER_TABLE}.")
else:
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "false")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"[Step 16] Incremental append: {rows_to_write:,} rows written to {SILVER_TABLE}.")

# ── Write all quarantine frames ───────────────────────────────────────────────
from functools import reduce

if quarantine_frames:
    dfs_to_quarantine = [rename_for_write(frame) for frame, _ in quarantine_frames]
    df_all_quarantine = reduce(
        lambda a, b: a.unionByName(b, allowMissingColumns=True),
        dfs_to_quarantine
    )

    for col_name in ["quarantine_reason", "quarantine_date"]:
        if col_name not in df_all_quarantine.columns:
            df_all_quarantine = df_all_quarantine.withColumn(col_name, F.lit(None).cast(T.StringType()))

    (
        df_all_quarantine.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(QUARANTINE_TABLE)
    )

    n_quarantine_total = df_all_quarantine.count()
    print(f"[Step 16] {n_quarantine_total:,} rows written to {QUARANTINE_TABLE}.")
else:
    n_quarantine_total = 0
    print("[Step 16] No rows to quarantine this run.")

dq_metrics["step_16_write"] = {
    "rows_written_to_silver":     rows_to_write,
    "rows_written_to_quarantine": n_quarantine_total,
}

StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 58, Finished, Available, Finished, False)

[Step 16] Incremental append: 1,145,019 rows written to silver_stg_nyc_311.
[Step 16] 41,789 rows written to silver_quarantine_311.


---
## Pre-Step C Output — Data-Quality Metrics Summary

All per-step metrics accumulated in `dq_metrics` are printed here as a summary
table and written to the Silver layer as a dated JSON audit record.  
One file is created per pipeline run, keyed by `TODAY`.  
Query these files over time to track pipeline health, catch regressions, and
verify that quarantine volumes are within expected bounds.

In [47]:
# ── Print summary ─────────────────────────────────────────────────────────────
print("\n" + "═" * 70)
print(f"  DATA QUALITY METRICS SUMMARY  —  Run date: {TODAY}")
print("═" * 70)

for step, metrics in dq_metrics.items():
    print(f"\n▸ {step}")
    for key, value in metrics.items():
        print(f"    {key:<45} {value}")

print("\n" + "═" * 70)

dq_metrics["_pipeline_summary"] = {
    "run_date":              TODAY,
    "ingested_from_bronze":  ingested_row_count,
    "written_to_silver":     dq_metrics.get("step_16_write", {}).get("rows_written_to_silver", "N/A"),
    "written_to_quarantine": dq_metrics.get("step_16_write", {}).get("rows_written_to_quarantine", "N/A"),
    "schema_drift_status":   dq_metrics.get("pre_A_schema_drift", {}).get("status", "N/A"),
    "row_count_match":       dq_metrics.get("pre_B_row_count", {}).get("match", "N/A"),
}

# ── Serialise and write DQ metrics to Delta ───────────────────────────────────
dq_json = json.dumps(dict(dq_metrics), indent=2, default=str)

dq_record = spark.createDataFrame(
    [(TODAY, dq_json)],
    schema=T.StructType([
        T.StructField("run_date",     T.StringType(), nullable=False),
        T.StructField("metrics_json", T.StringType(), nullable=False),
    ])
)

(
    dq_record.write
    .format("delta")
    .mode("append")
    .saveAsTable("dq_metrics_311")
)

print(f"\nDQ metrics written to dq_metrics_311 (run_date={TODAY})")
print("\nsilver_stg_nyc_311 is clean, typed, and queryable. Pipeline complete.")


StatementMeta(, e8dec4ca-54a3-48cc-a2ee-6bba16caba9b, 59, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  DATA QUALITY METRICS SUMMARY  —  Run date: 2026-04-20
══════════════════════════════════════════════════════════════════════

▸ pre_A_schema_drift
    expected_column_count                         42
    actual_column_count                           48
    missing_columns                               ['Location']
    extra_columns                                 ['descriptor_2', 'ingest_date', 'ingest_run_id', 'ingested_at', 'location_coordinates', 'open_data_channel_type', 'source']
    status                                        WARN

▸ pre_B_row_count
    ingested_row_count                            1207334
    manifest_count                                None
    match                                         True

▸ step_01_critical_nulls
    hard_rejected                                 0
    flagged_for_borough_recovery                  1068
    rows_remaining_in_df                          1207334

▸ 